# CRM Access Governance & Customer Data Protection
## Stage 6 — Data Classification & Privacy Engineering

This notebook extends the CRM Governance project into **customer data protection and privacy engineering**.

The current access-governance dataset does not contain real customer-level PII. Therefore, this stage creates a **synthetic customer layer** to simulate how a governed CRM analytics environment should handle personal data.

### Main Goals

1. Create a synthetic CRM customer dataset.
2. Classify fields as personal/non-personal data.
3. Distinguish direct identifiers, indirect identifiers, behavioral data, and governance attributes.
4. Define sensitivity levels.
5. Apply data minimization.
6. Apply masking and pseudonymization.
7. Create an analytics-safe customer layer.
8. Document privacy controls and their rationale.
9. Link privacy controls to the access-governance framework.
10. Prepare the project for future lineage, privacy monitoring, and AI Governance.

> **Important:** this is a simulated technical governance exercise. It does not constitute legal advice and does not claim that every field classification maps directly to an official LGPD/GDPR legal category.


In [1]:
# 1. Libraries and Settings

import pandas as pd
import numpy as np
import hashlib
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment ready.")


Environment ready.


# 2. Why a Synthetic Customer Layer Is Needed

The original dataset focuses on **who accesses the CRM and under which conditions**.

To demonstrate privacy engineering, we also need to simulate **what customer data the CRM contains**.

This stage introduces a synthetic customer table with attributes such as:

- customer identifier;
- name;
- email;
- phone;
- date of birth;
- region;
- signup date;
- marketing consent;
- customer segment;
- purchase metrics.

The objective is not to create realistic personal identities. The goal is to build a controlled environment for privacy-preserving transformations.


# 3. Create Synthetic Customer Data

In [2]:
n_customers = 10000

customer_ids = [f"CUST_{i:06d}" for i in range(1, n_customers + 1)]

first_names = [
    "Alex", "Jordan", "Taylor", "Morgan", "Casey",
    "Riley", "Jamie", "Avery", "Cameron", "Drew"
]

last_names = [
    "Silva", "Santos", "Oliveira", "Pereira", "Costa",
    "Rodrigues", "Almeida", "Nascimento", "Lima", "Souza"
]

states = ["SP", "RJ", "MG", "DF", "PR", "RS", "BA", "SC", "GO", "PE"]

segments = [
    "New Customer",
    "Regular",
    "High Value",
    "At Risk",
    "Inactive"
]

signup_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("2022-01-01", "2026-08-01", freq="D"),
        size=n_customers
    )
)

birth_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range("1955-01-01", "2005-12-31", freq="D"),
        size=n_customers
    )
)

customers = pd.DataFrame({
    "Customer_ID": customer_ids,
    "First_Name": np.random.choice(first_names, n_customers),
    "Last_Name": np.random.choice(last_names, n_customers),
    "Email": [
        f"customer{i}@example.com"
        for i in range(1, n_customers + 1)
    ],
    "Phone": [
        f"+55 11 9{np.random.randint(1000, 9999)}-{np.random.randint(1000, 9999)}"
        for _ in range(n_customers)
    ],
    "Birth_Date": birth_dates,
    "State": np.random.choice(states, n_customers),
    "Signup_Date": signup_dates,
    "Marketing_Consent": np.random.choice(
        [True, False],
        n_customers,
        p=[0.72, 0.28]
    ),
    "Customer_Segment": np.random.choice(
        segments,
        n_customers,
        p=[0.15, 0.45, 0.15, 0.15, 0.10]
    ),
    "Purchase_Count": np.random.poisson(8, n_customers),
    "Total_Revenue": np.round(
        np.random.gamma(2.5, 450, n_customers),
        2
    )
})

customers["Average_Ticket"] = np.where(
    customers["Purchase_Count"] > 0,
    customers["Total_Revenue"] / customers["Purchase_Count"],
    0
).round(2)

display(customers.head())
print(f"Rows: {len(customers):,}")


,Customer_ID,First_Name,Last_Name,Email,Phone,Birth_Date,State,Signup_Date,Marketing_Consent,Customer_Segment,Purchase_Count,Total_Revenue,Average_Ticket
0,CUST_000001,Jordan,Nascimento,customer1@example.com,+55 11 95047-1009,1990-04-13,PR,2025-01-31,True,Regular,11,873.34,79.39
1,CUST_000002,Jamie,Lima,customer2@example.com,+55 11 97463-6321,2000-12-09,SC,2025-12-30,True,High Value,16,3153.70,197.11
2,CUST_000003,Jordan,Santos,customer3@example.com,+55 11 95216-9865,2002-06-05,SC,2024-05-10,True,Regular,4,614.78,153.70
3,CUST_000004,Morgan,Lima,customer4@example.com,+55 11 98734-8506,1981-11-28,MG,2025-07-18,True,High Value,11,575.44,52.31
4,CUST_000005,Alex,Nascimento,customer5@example.com,+55 11 92456-1043,1961-11-15,GO,2025-02-04,True,High Value,4,1220.87,305.22


Rows: 10,000


# 4. Field-Level Privacy Classification

Fields are classified by their privacy role.

### Proposed Classes

- `Direct Identifier`
- `Indirect Identifier`
- `Behavioral / Transactional`
- `Governance / Consent`
- `Non-Personal Analytical Attribute`

This classification is designed for the project and is not intended as a legal taxonomy.


In [3]:
privacy_classification = pd.DataFrame([
    ["Customer_ID", "Direct Identifier", "Restricted", True, "Pseudonymize"],
    ["First_Name", "Direct Identifier", "Restricted", False, "Remove"],
    ["Last_Name", "Direct Identifier", "Restricted", False, "Remove"],
    ["Email", "Direct Identifier", "Restricted", False, "Mask or Remove"],
    ["Phone", "Direct Identifier", "Restricted", False, "Mask or Remove"],
    ["Birth_Date", "Indirect Identifier", "Confidential", False, "Generalize"],
    ["State", "Indirect Identifier", "Internal", True, "Keep"],
    ["Signup_Date", "Behavioral / Transactional", "Internal", True, "Keep"],
    ["Marketing_Consent", "Governance / Consent", "Restricted", True, "Keep"],
    ["Customer_Segment", "Behavioral / Transactional", "Confidential", True, "Keep"],
    ["Purchase_Count", "Behavioral / Transactional", "Confidential", True, "Keep"],
    ["Total_Revenue", "Behavioral / Transactional", "Confidential", True, "Keep"],
    ["Average_Ticket", "Behavioral / Transactional", "Confidential", True, "Keep"]
], columns=[
    "Field_Name",
    "Privacy_Class",
    "Sensitivity",
    "Required_for_Analytics",
    "Recommended_Treatment"
])

display(privacy_classification)


,Field_Name,Privacy_Class,Sensitivity,Required_for_Analytics,Recommended_Treatment
0,Customer_ID,Direct Identifier,Restricted,True,Pseudonymize
1,First_Name,Direct Identifier,Restricted,False,Remove
2,Last_Name,Direct Identifier,Restricted,False,Remove
3,Email,Direct Identifier,Restricted,False,Mask or Remove
4,Phone,Direct Identifier,Restricted,False,Mask or Remove
5,Birth_Date,Indirect Identifier,Confidential,False,Generalize
6,State,Indirect Identifier,Internal,True,Keep
7,Signup_Date,Behavioral / Transactional,Internal,True,Keep
8,Marketing_Consent,Governance / Consent,Restricted,True,Keep
9,Customer_Segment,Behavioral / Transactional,Confidential,True,Keep


# 5. Direct vs. Indirect Identifiers

### Direct Identifiers
Can directly point to an individual.

Examples:
- name;
- email;
- phone;
- customer identifier.

### Indirect Identifiers
May contribute to re-identification when combined with other attributes.

Examples:
- date of birth;
- location;
- rare demographic combinations.

This distinction helps determine whether a field should be removed, generalized, masked, or retained.


# 6. Data Minimization Matrix

The analytical layer should include only the data necessary for its intended purpose.

The proposed analytical use case is:

> Customer segmentation, CRM governance monitoring, consent analysis, and aggregate commercial analytics.

Fields that are not required for these purposes should not be exposed to the analytical layer.


In [4]:
minimization_matrix = privacy_classification[
    [
        "Field_Name",
        "Privacy_Class",
        "Sensitivity",
        "Required_for_Analytics",
        "Recommended_Treatment"
    ]
].copy()

display(minimization_matrix)


,Field_Name,Privacy_Class,Sensitivity,Required_for_Analytics,Recommended_Treatment
0,Customer_ID,Direct Identifier,Restricted,True,Pseudonymize
1,First_Name,Direct Identifier,Restricted,False,Remove
2,Last_Name,Direct Identifier,Restricted,False,Remove
3,Email,Direct Identifier,Restricted,False,Mask or Remove
4,Phone,Direct Identifier,Restricted,False,Mask or Remove
5,Birth_Date,Indirect Identifier,Confidential,False,Generalize
6,State,Indirect Identifier,Internal,True,Keep
7,Signup_Date,Behavioral / Transactional,Internal,True,Keep
8,Marketing_Consent,Governance / Consent,Restricted,True,Keep
9,Customer_Segment,Behavioral / Transactional,Confidential,True,Keep


# 7. Pseudonymization Function

The function below converts the original customer identifier into a deterministic pseudonymous token.

A deterministic hash allows the same customer to be linked across governed datasets without exposing the original identifier.

> In a real environment, secure key management, salts, HMAC, encryption, or tokenization services would normally be considered instead of plain hashing alone.


In [5]:
def pseudonymize_identifier(value, salt="crm-governance-demo"):
    raw_value = f"{salt}|{value}"

    return hashlib.sha256(
        raw_value.encode("utf-8")
    ).hexdigest()[:16]

customers["Customer_Token"] = (
    customers["Customer_ID"]
    .apply(pseudonymize_identifier)
)

display(
    customers[
        ["Customer_ID", "Customer_Token"]
    ].head()
)


,Customer_ID,Customer_Token
0,CUST_000001,7b91ee80bfbdf1d4
1,CUST_000002,f6b2732f9c4741b8
2,CUST_000003,a5018f78abdc7e1e
3,CUST_000004,0665cc2f3715bf80
4,CUST_000005,223e2270ecaedfcd


## Study Note — Hashing vs. Encryption

### Hashing
- one-way transformation;
- generally not intended to be reversed;
- useful for pseudonymous linking when designed correctly.

### Encryption
- reversible with the appropriate key;
- useful when authorized systems must recover the original value.

### Tokenization
- replaces the original value with a surrogate token;
- mapping is usually maintained in a protected token vault.

For this portfolio project, hashing is used only to demonstrate the concept of pseudonymization.


# 8. Email Masking

Masked values preserve limited operational context while reducing direct exposure.


In [6]:
def mask_email(email):
    if pd.isna(email) or "@" not in email:
        return None

    local, domain = email.split("@", 1)

    if len(local) <= 1:
        masked_local = "*"
    else:
        masked_local = local[0] + "***"

    return f"{masked_local}@{domain}"

customers["Email_Masked"] = (
    customers["Email"]
    .apply(mask_email)
)

display(
    customers[
        ["Email", "Email_Masked"]
    ].head()
)


,Email,Email_Masked
0,customer1@example.com,c***@example.com
1,customer2@example.com,c***@example.com
2,customer3@example.com,c***@example.com
3,customer4@example.com,c***@example.com
4,customer5@example.com,c***@example.com


# 9. Phone Masking

In [7]:
def mask_phone(phone):
    if pd.isna(phone):
        return None

    digits = "".join(
        char for char in str(phone)
        if char.isdigit()
    )

    if len(digits) < 4:
        return "****"

    return "*" * (len(digits) - 4) + digits[-4:]

customers["Phone_Masked"] = (
    customers["Phone"]
    .apply(mask_phone)
)

display(
    customers[
        ["Phone", "Phone_Masked"]
    ].head()
)


,Phone,Phone_Masked
0,+55 11 95047-1009,*********1009
1,+55 11 97463-6321,*********6321
2,+55 11 95216-9865,*********9865
3,+55 11 98734-8506,*********8506
4,+55 11 92456-1043,*********1043


# 10. Date Generalization

Exact dates of birth are usually unnecessary for aggregate CRM analytics.

We derive `Age_Group` instead of exposing the exact birth date.


In [8]:
REFERENCE_DATE = pd.Timestamp("2026-08-25")

customers["Age"] = (
    (
        REFERENCE_DATE -
        customers["Birth_Date"]
    ).dt.days / 365.25
).astype(int)

age_bins = [0, 24, 34, 44, 54, 64, 200]

age_labels = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

customers["Age_Group"] = pd.cut(
    customers["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

display(
    customers[
        ["Birth_Date", "Age", "Age_Group"]
    ].head()
)


,Birth_Date,Age,Age_Group
0,1990-04-13,36,35-44
1,2000-12-09,25,25-34
2,2002-06-05,24,18-24
3,1981-11-28,44,35-44
4,1961-11-15,64,55-64


# 11. Build the Analytics-Safe Customer Layer

The analytics-safe layer removes direct identifiers that are unnecessary for analytical use.

### Removed
- Customer_ID
- First_Name
- Last_Name
- Email
- Phone
- Birth_Date
- exact Age

### Retained or Derived
- Customer_Token
- Age_Group
- State
- Signup_Date
- Marketing_Consent
- Customer_Segment
- Purchase metrics


In [9]:
analytics_safe_customer = customers[
    [
        "Customer_Token",
        "Age_Group",
        "State",
        "Signup_Date",
        "Marketing_Consent",
        "Customer_Segment",
        "Purchase_Count",
        "Total_Revenue",
        "Average_Ticket"
    ]
].copy()

display(analytics_safe_customer.head())


,Customer_Token,Age_Group,State,Signup_Date,Marketing_Consent,Customer_Segment,Purchase_Count,Total_Revenue,Average_Ticket
0,7b91ee80bfbdf1d4,35-44,PR,2025-01-31,True,Regular,11,873.34,79.39
1,f6b2732f9c4741b8,25-34,SC,2025-12-30,True,High Value,16,3153.70,197.11
2,a5018f78abdc7e1e,18-24,SC,2024-05-10,True,Regular,4,614.78,153.70
3,0665cc2f3715bf80,35-44,MG,2025-07-18,True,High Value,11,575.44,52.31
4,223e2270ecaedfcd,55-64,GO,2025-02-04,True,High Value,4,1220.87,305.22


# 12. Exposure Comparison

This section compares the raw and analytics-safe layers.


In [10]:
raw_columns = set(customers.columns)

safe_columns = set(
    analytics_safe_customer.columns
)

removed_from_analytics = sorted(
    raw_columns - safe_columns
)

print("Fields excluded from the analytics-safe layer:")
for field in removed_from_analytics:
    print("-", field)


Fields excluded from the analytics-safe layer:
- Age
- Birth_Date
- Customer_ID
- Email
- Email_Masked
- First_Name
- Last_Name
- Phone
- Phone_Masked


# 13. Privacy Control Catalog

Each privacy control receives an ID and rationale to make the framework auditable.


In [11]:
privacy_control_catalog = pd.DataFrame([
    [
        "PRIV-001",
        "Data Minimization",
        "Remove customer names from analytics layer",
        "First_Name, Last_Name",
        "REMOVE",
        "Names are not required for aggregate analytics."
    ],
    [
        "PRIV-002",
        "Pseudonymization",
        "Replace Customer_ID with Customer_Token",
        "Customer_ID",
        "PSEUDONYMIZE",
        "Allows analytical linkage without exposing the source identifier."
    ],
    [
        "PRIV-003",
        "Masking",
        "Mask customer email",
        "Email",
        "MASK",
        "Reduces direct identifier exposure where limited display is needed."
    ],
    [
        "PRIV-004",
        "Masking",
        "Mask customer phone",
        "Phone",
        "MASK",
        "Reduces direct identifier exposure."
    ],
    [
        "PRIV-005",
        "Generalization",
        "Replace exact birth date with age group",
        "Birth_Date",
        "GENERALIZE",
        "Exact date of birth is unnecessary for aggregate analytics."
    ],
    [
        "PRIV-006",
        "Consent Governance",
        "Retain marketing consent as a controlled governance attribute",
        "Marketing_Consent",
        "KEEP / RESTRICT",
        "Consent status is required for governance and marketing eligibility analysis."
    ],
    [
        "PRIV-007",
        "Purpose Limitation",
        "Expose only analytics-required fields to the analytical layer",
        "Multiple Fields",
        "MINIMIZE",
        "Limits access to data necessary for the analytical use case."
    ]
], columns=[
    "Control_ID",
    "Control_Type",
    "Control_Description",
    "Field",
    "Treatment",
    "Rationale"
])

display(privacy_control_catalog)


,Control_ID,Control_Type,Control_Description,Field,Treatment,Rationale
0,PRIV-001,Data Minimization,Remove customer names from analytics layer,"First_Name, Last_Name",REMOVE,Names are not required for aggregate analytics.
1,PRIV-002,Pseudonymization,Replace Customer_ID with Customer_Token,Customer_ID,PSEUDONYMIZE,Allows analytical linkage without exposing the...
2,PRIV-003,Masking,Mask customer email,Email,MASK,Reduces direct identifier exposure where limit...
3,PRIV-004,Masking,Mask customer phone,Phone,MASK,Reduces direct identifier exposure.
4,PRIV-005,Generalization,Replace exact birth date with age group,Birth_Date,GENERALIZE,Exact date of birth is unnecessary for aggrega...
5,PRIV-006,Consent Governance,Retain marketing consent as a controlled gover...,Marketing_Consent,KEEP / RESTRICT,Consent status is required for governance and ...
6,PRIV-007,Purpose Limitation,Expose only analytics-required fields to the a...,Multiple Fields,MINIMIZE,Limits access to data necessary for the analyt...


# 14. Privacy Classification Coverage

A privacy program should also measure whether all fields have been classified.


In [12]:
source_fields = set(
    [
        "Customer_ID",
        "First_Name",
        "Last_Name",
        "Email",
        "Phone",
        "Birth_Date",
        "State",
        "Signup_Date",
        "Marketing_Consent",
        "Customer_Segment",
        "Purchase_Count",
        "Total_Revenue",
        "Average_Ticket"
    ]
)

classified_fields = set(
    privacy_classification["Field_Name"]
)

unclassified_fields = sorted(
    source_fields - classified_fields
)

classification_coverage = (
    len(classified_fields.intersection(source_fields))
    /
    len(source_fields)
    * 100
)

print(
    f"Privacy classification coverage: "
    f"{classification_coverage:.2f}%"
)

print("Unclassified fields:", unclassified_fields)


Privacy classification coverage: 100.00%
Unclassified fields: []


# 15. Sensitive Field Inventory

In [13]:
sensitive_inventory = (
    privacy_classification.groupby(
        ["Sensitivity", "Privacy_Class"]
    )
    .agg(
        Fields=("Field_Name", "count")
    )
    .reset_index()
)

display(sensitive_inventory)


,Sensitivity,Privacy_Class,Fields
0,Confidential,Behavioral / Transactional,4
1,Confidential,Indirect Identifier,1
2,Internal,Behavioral / Transactional,1
3,Internal,Indirect Identifier,1
4,Restricted,Direct Identifier,5
5,Restricted,Governance / Consent,1


# 16. Analytics Necessity Review

This view identifies fields that are sensitive but still required for analytics.


In [14]:
analytics_necessity = privacy_classification[
    privacy_classification[
        "Required_for_Analytics"
    ] == True
][
    [
        "Field_Name",
        "Privacy_Class",
        "Sensitivity",
        "Recommended_Treatment"
    ]
]

display(analytics_necessity)


,Field_Name,Privacy_Class,Sensitivity,Recommended_Treatment
0,Customer_ID,Direct Identifier,Restricted,Pseudonymize
6,State,Indirect Identifier,Internal,Keep
7,Signup_Date,Behavioral / Transactional,Internal,Keep
8,Marketing_Consent,Governance / Consent,Restricted,Keep
9,Customer_Segment,Behavioral / Transactional,Confidential,Keep
10,Purchase_Count,Behavioral / Transactional,Confidential,Keep
11,Total_Revenue,Behavioral / Transactional,Confidential,Keep
12,Average_Ticket,Behavioral / Transactional,Confidential,Keep


# 17. Privacy Risk Flags at Field Level

A simple field-level governance flag helps prioritize privacy engineering effort.


In [15]:
privacy_risk_weights = {
    "Internal": 1,
    "Confidential": 2,
    "Restricted": 3
}

privacy_classification["Sensitivity_Weight"] = (
    privacy_classification[
        "Sensitivity"
    ].map(privacy_risk_weights)
)

privacy_classification["Direct_Identifier_Flag"] = (
    privacy_classification[
        "Privacy_Class"
    ].eq("Direct Identifier")
)

privacy_classification["Privacy_Priority_Score"] = (
    privacy_classification[
        "Sensitivity_Weight"
    ]
    +
    privacy_classification[
        "Direct_Identifier_Flag"
    ].astype(int) * 2
)

display(
    privacy_classification.sort_values(
        "Privacy_Priority_Score",
        ascending=False
    )
)


,Field_Name,Privacy_Class,Sensitivity,Required_for_Analytics,Recommended_Treatment,Sensitivity_Weight,Direct_Identifier_Flag,Privacy_Priority_Score
0,Customer_ID,Direct Identifier,Restricted,True,Pseudonymize,3,True,5
1,First_Name,Direct Identifier,Restricted,False,Remove,3,True,5
2,Last_Name,Direct Identifier,Restricted,False,Remove,3,True,5
3,Email,Direct Identifier,Restricted,False,Mask or Remove,3,True,5
4,Phone,Direct Identifier,Restricted,False,Mask or Remove,3,True,5
8,Marketing_Consent,Governance / Consent,Restricted,True,Keep,3,False,3
5,Birth_Date,Indirect Identifier,Confidential,False,Generalize,2,False,2
10,Purchase_Count,Behavioral / Transactional,Confidential,True,Keep,2,False,2
9,Customer_Segment,Behavioral / Transactional,Confidential,True,Keep,2,False,2
11,Total_Revenue,Behavioral / Transactional,Confidential,True,Keep,2,False,2


# 18. Link Privacy Controls to Access Governance

Privacy protection and access governance should reinforce each other.

Examples:

- pseudonymized customer data can be exposed more broadly than direct identifiers;
- restricted fields should require stronger role/action controls;
- exports involving direct identifiers should receive stricter treatment;
- analytics users should consume the safe layer instead of raw CRM records.


In [16]:
privacy_access_mapping = pd.DataFrame([
    [
        "Restricted Direct Identifiers",
        "Admin / Authorized Operations Only",
        "Raw CRM Layer",
        "BLOCK or REVIEW for broad export"
    ],
    [
        "Masked Contact Data",
        "Limited Operational Roles",
        "Protected Operational Layer",
        "REVIEW for bulk access"
    ],
    [
        "Pseudonymized Customer Data",
        "Analytics / Governance Roles",
        "Analytics-Safe Layer",
        "ALLOW under approved purpose"
    ],
    [
        "Aggregate Customer Metrics",
        "Analytics / Management",
        "Analytics Layer",
        "ALLOW under normal governance"
    ]
], columns=[
    "Data_Category",
    "Recommended_Access",
    "Data_Layer",
    "Governance_Treatment"
])

display(privacy_access_mapping)


,Data_Category,Recommended_Access,Data_Layer,Governance_Treatment
0,Restricted Direct Identifiers,Admin / Authorized Operations Only,Raw CRM Layer,BLOCK or REVIEW for broad export
1,Masked Contact Data,Limited Operational Roles,Protected Operational Layer,REVIEW for bulk access
2,Pseudonymized Customer Data,Analytics / Governance Roles,Analytics-Safe Layer,ALLOW under approved purpose
3,Aggregate Customer Metrics,Analytics / Management,Analytics Layer,ALLOW under normal governance


# 19. Proposed Privacy Architecture

```text
RAW CRM CUSTOMER DATA
        |
        v
PII / PRIVACY CLASSIFICATION
        |
        v
DATA MINIMIZATION
        |
  +-----+-------------------+
  |                         |
  v                         v
MASK / PSEUDONYMIZE      REMOVE
  |                         |
  +-----------+-------------+
              |
              v
      ANALYTICS-SAFE LAYER
              |
              v
      GOVERNED ANALYTICS
```

The original direct identifiers remain restricted to the operational CRM layer.


# 20. Privacy Monitoring KPIs

Future governance monitoring can include:

- Privacy Classification Coverage %
- Restricted Fields
- Direct Identifier Count
- Fields Removed from Analytics
- Fields Masked
- Fields Pseudonymized
- Analytics-Safe Fields
- Consent Coverage %
- Customers Without Marketing Consent
- Privacy Controls Implemented
- Privacy Control Exceptions


In [17]:
privacy_kpis = pd.DataFrame({
    "Metric": [
        "Customer Records",
        "Classified Source Fields",
        "Direct Identifier Fields",
        "Restricted Fields",
        "Analytics-Safe Fields",
        "Marketing Consent Rate"
    ],
    "Value": [
        len(customers),
        len(classified_fields),
        privacy_classification[
            "Privacy_Class"
        ].eq("Direct Identifier").sum(),
        privacy_classification[
            "Sensitivity"
        ].eq("Restricted").sum(),
        analytics_safe_customer.shape[1],
        customers[
            "Marketing_Consent"
        ].mean() * 100
    ]
})

display(privacy_kpis.round(2))


,Metric,Value
0,Customer Records,10000.00
1,Classified Source Fields,13.00
2,Direct Identifier Fields,5.00
3,Restricted Fields,6.00
4,Analytics-Safe Fields,9.00
5,Marketing Consent Rate,70.91


# 21. Exportable Privacy Artifacts

The following DataFrames are intended to become persistent project artifacts:

- `customers`
- `privacy_classification`
- `minimization_matrix`
- `analytics_safe_customer`
- `privacy_control_catalog`
- `privacy_access_mapping`
- `privacy_kpis`

Suggested GitHub structure:

```text
governance/
├── privacy_classification.csv
├── privacy_control_catalog.csv
├── privacy_access_mapping.csv
└── data_minimization_matrix.csv

data/
├── raw/
│   └── synthetic_customers.csv
└── processed/
    └── analytics_safe_customers.csv
```


# 22. Findings to Document

## Privacy Classification

- **Total customer fields:** 13 source customer fields were classified.

- **Direct identifiers:** 5 — `Customer_ID`, `First_Name`, `Last_Name`, `Email`, and `Phone`.

- **Indirect identifiers:** 2 — `Birth_Date` and `State`.

- **Restricted fields:** 6 — the 5 direct identifiers plus `Marketing_Consent`.

- **Classification coverage:** 100%. All 13 source customer fields received a privacy classification.

## Minimization

- **Fields removed:** 2 direct identifiers are explicitly removed from the analytics layer: `First_Name` and `Last_Name`.

- **Fields retained:** 7 source fields are retained directly for analytical use: `State`, `Signup_Date`, `Marketing_Consent`, `Customer_Segment`, `Purchase_Count`, `Total_Revenue`, and `Average_Ticket`.

- **Fields transformed:** 4 source fields receive privacy-preserving treatment before analytical use: `Customer_ID` is pseudonymized, `Email` and `Phone` are masked when limited operational display is required, and `Birth_Date` is generalized into `Age_Group`.

## Privacy Engineering

- **Pseudonymized fields:** 1 — `Customer_ID`, replaced by `Customer_Token`.

- **Masked fields:** 2 — `Email` and `Phone`.

- **Generalized fields:** 1 — `Birth_Date`, converted into `Age_Group`.

## Consent

- **Marketing consent rate:** 70.91%.

- **Customers without consent:** 2,909 of 10,000 customers.

## Governance Conclusions

1. **The privacy classification provides full coverage of the customer dataset and clearly distinguishes direct identifiers, indirect identifiers, behavioral attributes, and consent-related data, allowing privacy controls to be applied according to the role and sensitivity of each field.**

2. **The analytics-safe layer applies data minimization and privacy engineering by removing unnecessary direct identifiers and replacing sensitive source fields with pseudonymized, masked, or generalized representations where appropriate.**

3. **Marketing consent must remain a governed attribute rather than merely an analytical variable, because consent status directly affects whether a customer can be included in marketing-related processing and should therefore be monitored, restricted, and auditable.**

# 23. Limitations

1. Customer data is fully synthetic.
2. No real legal assessment is performed.
3. Hashing is used only to demonstrate pseudonymization concepts.
4. Secure key management and token vaults are not implemented.
5. No real retention policy is available.
6. No subject-rights workflow is implemented yet.
7. No automated PII scanner is used.
8. Privacy classifications are project-level governance classifications.


# 24. Next Step — Data Lineage & Traceability

## Stage 7 — Data Lineage & Traceability

Planned outputs:

- source-to-target mapping;
- transformation inventory;
- field-level lineage;
- relationship between source data and governance rules;
- relationship between privacy controls and analytical outputs;
- traceability from raw CRM to Power BI metrics;
- lineage documentation for GitHub.

This stage will make the project easier to audit and explain end to end.
